# baseline_v0.6 — YOLO11 단일 모델 + 데이터 정합성 전처리 + 출력 파일 관리

v0.5에서 추가/변경:
- Drive `outputs/` 복사 단계 제거 (매번 새 학습 가정 — Drive에는 백업만 쌓임)
- 학습 결과 파일명/저장위치를 한 셀에서 설정 가능 (`RUN_NAME`, `SUBMISSION_NAME`)
- `하이퍼파라미터_설명서.md` 동봉

**사전 준비:** 런타임 → 런타임 유형 변경 → **GPU (T4)**

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 경로 & 출력 파일 설정 — **이 셀만 수정하세요**

- `DRIVE_PROJECT_DIR`: 구글 드라이브의 프로젝트 루트
- `RUN_NAME`: 이번 학습 실행 이름. `outputs/yolo/<RUN_NAME>/` 아래에 가중치/그래프가 저장됨
- `SUBMISSION_NAME`: 캐글 제출용 CSV 파일 이름 (기본 `submission_yolo.csv`)

### 학습/추론 후 생기는 파일들 (로컬 `/content/project/baseline/` 기준)
| 경로 | 내용 |
| --- | --- |
| `data/processed/annotations.json`, `class_mapping.json` | 어노테이션 전처리 결과 |
| `data/yolo/{images,labels}/{train,val,test}/` | YOLO 포맷 변환 데이터 |
| `data/yolo/dataset.yaml`, `orig_id_map.json` | YOLO 데이터 정의 / 클래스ID 복원 매핑 |
| `outputs/yolo/<RUN_NAME>/weights/best.pt`, `last.pt` | 학습된 모델 가중치 |
| `outputs/yolo/<RUN_NAME>/results.csv`, `*.png` | 학습 곡선/지표 |
| `outputs/predictions/<SUBMISSION_NAME>` | **캐글 제출용 CSV** |

마지막 셀(Drive 백업)에서 위 중 `outputs/yolo/`와 `outputs/predictions/`만 Drive로 올라갑니다.

In [ ]:
import os

# ─── 여기만 수정 ──────────────────────────────────────────────────
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/베이스라인_코랩버전v1.4'

# 학습 실행 / 출력 파일 이름
RUN_NAME        = 'test_run'            # outputs/yolo/<RUN_NAME>/
SUBMISSION_NAME = 'submission_yolo.csv' # 캐글 제출용 CSV 이름
# ────────────────────────────────────────────────────────────────

DRIVE_BASELINE   = os.path.join(DRIVE_PROJECT_DIR, 'baseline')
DRIVE_OUTPUTS    = os.path.join(DRIVE_PROJECT_DIR, 'outputs')
DRIVE_DATA_ZIP   = os.path.join(DRIVE_PROJECT_DIR, 'sprint_ai_project1_data.zip')

LOCAL_PROJECT_DIR = '/content/project'
LOCAL_BASELINE    = os.path.join(LOCAL_PROJECT_DIR, 'baseline')
LOCAL_OUTPUTS     = os.path.join(LOCAL_BASELINE, 'outputs')
LOCAL_DATA_DIR    = os.path.join(LOCAL_PROJECT_DIR, 'sprint_ai_project1_data')

# 학습/추론 결과 경로 (위 RUN_NAME / SUBMISSION_NAME 에서 파생)
TRAIN_OUTPUT_DIR = f'outputs/yolo/{RUN_NAME}'
BEST_WEIGHTS     = f'{TRAIN_OUTPUT_DIR}/weights/best.pt'
SUBMISSION_PATH  = f'outputs/predictions/{SUBMISSION_NAME}'

print('--- Drive ---')
for p in [DRIVE_PROJECT_DIR, DRIVE_BASELINE, DRIVE_DATA_ZIP]:
    print(f'  exists={os.path.exists(p)}  {p}')
print('--- Local ---')
for p in [LOCAL_PROJECT_DIR, LOCAL_BASELINE, LOCAL_DATA_DIR]:
    print(f'  exists={os.path.exists(p)}  {p}')
print('--- Run config ---')
print(f'  RUN_NAME        = {RUN_NAME}')
print(f'  SUBMISSION_NAME = {SUBMISSION_NAME}')

## 3. Drive → Colab 로컬 복제

- baseline: 그대로 복사
- 데이터셋: zip 한 덩어리로 받아 로컬 unzip
- **outputs는 복사 안 함** (매번 새 학습 가정 — Drive에는 11번 셀에서 백업만 함)
- 이미 있으면 skip (재실행 안전)

In [ ]:
import os, shutil, subprocess, time

os.makedirs(LOCAL_PROJECT_DIR, exist_ok=True)

def copy_dir(src, dst, label):
    if not os.path.exists(src):
        print(f'[skip] {label}: Drive에 없음')
        return
    if os.path.exists(dst):
        print(f'[skip] {label}: 이미 로컬에 있음')
        return
    t0 = time.time()
    shutil.copytree(src, dst)
    print(f'[ok]   {label} 복사 완료 ({time.time()-t0:.1f}s)')

copy_dir(DRIVE_BASELINE, LOCAL_BASELINE, 'baseline/')

if os.path.exists(LOCAL_DATA_DIR):
    print(f'[skip] dataset: 이미 해제됨')
else:
    assert os.path.exists(DRIVE_DATA_ZIP), f'데이터 zip 없음: {DRIVE_DATA_ZIP}'
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    print('[..] dataset 해제 중 (수 분 소요)...')
    t0 = time.time()
    subprocess.run(['unzip', '-q', DRIVE_DATA_ZIP, '-d', LOCAL_DATA_DIR], check=True)
    print(f'[ok] dataset 해제 완료 ({time.time()-t0:.1f}s) → {LOCAL_DATA_DIR}')
    print('     하위:', sorted(os.listdir(LOCAL_DATA_DIR))[:5])

## 4. 작업 디렉토리 + config 경로 갱신

In [ ]:
import os, yaml

os.chdir(LOCAL_BASELINE)
print('CWD:', os.getcwd())

cfg_path = 'configs/default.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

cfg['data']['data_root']     = LOCAL_DATA_DIR
cfg['data']['processed_dir'] = './data/processed'
cfg['data']['num_workers']   = 2

with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True, default_flow_style=False)

print('data_root:', cfg['data']['data_root'])

## 5. 패키지 설치 (베이스 + YOLO)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q ultralytics

## 6. GPU 확인

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 6-b. 데이터 정합성 필터 — bbox 수가 안 맞는 이미지 제거

이미지 파일명 prefix(예: `K-001900-003544-004543-016548_..._.png`)에서 첫 `_` 앞의 `-` 개수가
실제 알약 개수입니다. 같은 이미지에 대한 bbox JSON 파일 수의 합과 일치해야 정상.
불일치하는 이미지와 해당 JSON들을 **로컬에서만** 삭제합니다.
(Drive의 zip 원본은 그대로. 재실행 시 unzip 하면 원상복구)

In [ ]:
import os
from pathlib import Path

TRAIN_IMG_DIR = Path(LOCAL_DATA_DIR) / 'train_images'
TRAIN_ANN_DIR = Path(LOCAL_DATA_DIR) / 'train_annotations'

images = sorted(TRAIN_IMG_DIR.glob('*.png'))
print(f'[scan] {len(images)} images')

def expected_count(fname: str) -> int:
    # 'K-001900-003544-004543-016548_0_2_0_2_...png' -> 첫 _ 앞 대시 개수
    return fname.split('_', 1)[0].count('-')

def find_json_files(img_path: Path):
    # train_annotations/<prefix>_json/<K-XXXXXX>/<imagename>.json
    prefix = img_path.name.split('_', 1)[0]
    ann_dir = TRAIN_ANN_DIR / f'{prefix}_json'
    stem = img_path.stem
    if not ann_dir.exists():
        return []
    return list(ann_dir.glob(f'*/{stem}.json'))

dropped_imgs, dropped_jsons = [], 0
kept = 0
for img in images:
    exp = expected_count(img.name)
    jsons = find_json_files(img)
    got = len(jsons)
    if exp == got:
        kept += 1
        continue
    # 삭제
    img.unlink()
    for j in jsons:
        j.unlink()
        dropped_jsons += 1
    dropped_imgs.append((img.name, exp, got))

print(f'kept:    {kept}')
print(f'dropped: {len(dropped_imgs)} images, {dropped_jsons} json files')
for fn, exp, got in dropped_imgs[:20]:
    print(f'  expected={exp} got={got}  {fn}')
if len(dropped_imgs) > 20:
    print(f'  ... and {len(dropped_imgs)-20} more')

## 7. 어노테이션 전처리 (최초 1회)

In [ ]:
!python scripts/preprocess.py

## 8. COCO → YOLO 형식 변환 (최초 1회)

In [ ]:
!python scripts/convert_to_yolo.py

## 9. YOLO11 학습 (단일 모델)

동작 확인 우선 → 가벼운 설정으로 짧게. 본학습 시 `--epochs 60 --batch 8 --imgsz 1280` 권장.

In [ ]:
!python train_yolo.py \
    --model yolo11m.pt \
    --epochs 3 \
    --batch 8 \
    --imgsz 640 \
    --output outputs/yolo \
    --name {RUN_NAME}

## 10. YOLO11 추론 → 캐글 제출 CSV

inference_yolo.py는 항상 `outputs/predictions/submission_yolo.csv`로 저장하므로,
추론 후 `SUBMISSION_NAME`이 기본값과 다르면 자동으로 이름을 바꿔줍니다.

In [ ]:
import os

!python inference_yolo.py \
    --checkpoint {BEST_WEIGHTS} \
    --conf 0.2

default_csv = 'outputs/predictions/submission_yolo.csv'
if SUBMISSION_PATH != default_csv and os.path.exists(default_csv):
    os.replace(default_csv, SUBMISSION_PATH)
    print(f'[rename] {default_csv} → {SUBMISSION_PATH}')

In [ ]:
import os, pandas as pd
csv_path = SUBMISSION_PATH
assert os.path.exists(csv_path), f'CSV 없음: {csv_path}'
df = pd.read_csv(csv_path)
print('총 예측 수:', len(df))
print('이미지 수:', df['image_id'].nunique())
df.head()

## 10-b. 예측 결과 시각화

테스트 이미지 몇 장을 골라 예측 박스를 그려봅니다.
박스가 알약 위에 잘 얹혀있고, score가 합리적이면 (~0.3~0.95) 잘 동작하는 것.

In [ ]:
import os, random, pandas as pd
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# ─── 조절 가능 ──────────────────────────
N_IMAGES = 6
SEED     = 0
# ───────────────────────────────────────

TEST_DIR = os.path.join(LOCAL_DATA_DIR, 'test_images')
df = pd.read_csv(SUBMISSION_PATH)

if SEED is not None:
    random.seed(SEED)
pred_ids = sorted(df['image_id'].unique().tolist())
picked = random.sample(pred_ids, min(N_IMAGES, len(pred_ids)))

cols = 3
rows = (len(picked) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
axes = axes.flatten() if rows*cols > 1 else [axes]

for i, (ax, img_id) in enumerate(zip(axes, picked), start=1):
    img_path = os.path.join(TEST_DIR, f'{img_id}.png')
    if not os.path.exists(img_path):
        ax.set_title(f'[{i}] (file not found)'); ax.axis('off'); continue

    img = Image.open(img_path).convert('RGB').copy()
    draw = ImageDraw.Draw(img)
    rows_for_img = df[df['image_id'] == img_id]
    for _, r in rows_for_img.iterrows():
        x1, y1 = r['bbox_x'], r['bbox_y']
        x2, y2 = x1 + r['bbox_w'], y1 + r['bbox_h']
        draw.rectangle([x1, y1, x2, y2], outline='red', width=4)
        draw.text((x1+3, max(0, y1-14)), f"cls={int(r['category_id'])} {r['score']:.2f}", fill='red')
    ax.imshow(img)
    ax.set_title(f'[{i}]  boxes: {len(rows_for_img)}')
    ax.axis('off')

for ax in axes[len(picked):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

print('\n--- image_ids ---')
for i, img_id in enumerate(picked, start=1):
    print(f'[{i}] {img_id}')

## 11. Drive 백업

In [ ]:
import shutil, os, time

os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

for folder in ['predictions', 'yolo']:
    src = os.path.join(LOCAL_OUTPUTS, folder)
    dst = os.path.join(DRIVE_OUTPUTS, folder)
    if not os.path.exists(src):
        continue
    t0 = time.time()
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'[ok] {folder} → Drive ({time.time()-t0:.1f}s)')

print('백업 완료:', DRIVE_OUTPUTS)